# NOTE: EXPERIMENTAL STATUS
**The Taigi POJ ASR model used in this notebook is currently in an experimental phase and is considered immature. Accuracy and performance may vary significantly depending on the input audio.**

# Fast-Whisper Interactive Inference (Gradio Version)

This notebook provides a Gradio interface for fast inference using `fast-whisper`. It supports loading native CTranslate2 (CT2) models or Hugging Face models (with optional PEFT adapters) by converting them on-the-fly.

In [2]:
# --- 🌐 ENVIRONMENT & AUTHENTICATION ---

INSTALL_IN_LOCAL = False  # Set to True to force installing dependencies locally

import os, sys, torch, warnings
from pathlib import Path
import urllib.request

# Environment Detection
def _is_colab() -> bool:
    try:
        import google.colab
        return True
    except ImportError:
        return False

def _is_kaggle() -> bool:
    return os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None

IS_COLAB  = _is_colab()
IS_KAGGLE = _is_kaggle()
IS_LOCAL  = not IS_COLAB and not IS_KAGGLE

# 1. Dependency Management
dependencies = ["faster-whisper", "ctranslate2", "transformers", "peft", "accelerate", "librosa", "gradio"]

if IS_KAGGLE or IS_COLAB:
    print(f"🌐 Running on Cloud. Installing dependencies: {', '.join(dependencies)}")
    !pip install {' '.join(dependencies)}
    print(f"💻 Running on Local System. Skipping dependency installation (ensure they are installed in your venv).")
elif IS_LOCAL and INSTALL_IN_LOCAL:
    print(f"💻 Running on Local System. Forcing dependency installation...")
    !pip install transformers peft numpy datasets bitsandbytes accelerate librosa gradio
else:
    print(f"💻 Running on Local System. Skipping dependency installation.")

# --- 2. Script Downloader ---
script_dir = "./scripts"
script_path = os.path.join(script_dir, "poj_converter.py")
os.makedirs(script_dir, exist_ok=True)
if not os.path.exists(script_path):
    url = "https://github.com/CyberOoHim/utils/raw/refs/heads/main/taigi/poj_number_diacritics_converter_2026_0327.py"     
    try:
        urllib.request.urlretrieve(url, script_path)
        print("✅ Downloaded poj_converter.py")
    except Exception as e:
        print(f"⚠️ Failed to download poj_converter.py: {e}")

# 3. Hugging Face Authentication
HF_TOKEN = None
if IS_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
elif IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
else:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("✅ Hugging Face authentication successful.")
else:
    print("🔓 No HF_TOKEN found. Using public repository access.")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


💻 Running on Local System. Skipping dependency installation.
🔓 No HF_TOKEN found. Using public repository access.
Using device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [ ]:
# @title --- 🛠️ CONFIGURATION --- { display-mode: "both" }
model_path = "lazy-worm/whisper-medium-ct2-taigi-unified-part-3-r5-adapter-20260425-234456" # @param ["lazy-worm/whisper-medium-ct2-taigi-unified-part-3-r3-20260419-155233", "lazy-worm/whisper-medium-ct2-taigi-unified-part-3-r5-adapter-20260425-234456"]
processor_path = "openai/whisper-medium" # @param {type:"string"}
adapter_path = "" # @param {type:"string"}
model_format = "ct2" # @param ["ct2", "hf"]
language = "nan" # @param {type:"string"}
numeric_to_poj = True # @param {type:"boolean"}

# Override from .env if local
if IS_LOCAL and os.getenv("model_path"):
    model_path = os.getenv("model_path")

# Status Summary
env_name = "Local" if IS_LOCAL else "Cloud (Colab/Kaggle)"
print(f"🌍 Environment: {env_name}")
print(f"📦 Model Path: {model_path}")
print()


🌍 Environment: Local
📦 Model Path: C:\Users\marti\Projects\taigi_fast_whisper_finetuner\weights\whisper-medium-ct2-taigi-unified-part-3-r1-adapter-20260419-155233



In [4]:
import warnings
import numpy as np
import torch
import librosa
from pathlib import Path
from transformers import WhisperProcessor, WhisperForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel
import ctranslate2
from faster_whisper import WhisperModel
import shutil
import sys
import os

warnings.filterwarnings("ignore")

# 1. Setup POJ Converter
_proj_root = Path(".").resolve()
script_path = _proj_root / "scripts"
if script_path.exists() and str(script_path) not in sys.path:
    sys.path.append(str(script_path))

try:
    import poj_converter
    POJ_AVAILABLE = True
except ImportError:
    POJ_AVAILABLE = False
    print("⚠️ POJ converter not found.")

def load_and_convert_model():
    global model, processor
    
    if 'model' in globals():
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if model_format == "ct2":
        print(f"⏳ Loading native CT2 model from: {model_path}...")
        compute = "float16" if device == "cuda" else "int8"
        model = WhisperModel(model_path, device=device, compute_type=compute)
        # Use processor_path for CT2 models as they lack processor files
        processor = WhisperProcessor.from_pretrained(processor_path, language=None)
    
    elif model_format == "hf":
        print(f"⏳ Loading HF model: {model_path}...")
        inf_device = "cuda" if torch.cuda.is_available() else "cpu"
        inf_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

        processor = WhisperProcessor.from_pretrained(model_path, language=None)
        hf_model = WhisperForConditionalGeneration.from_pretrained(
            model_path, 
            torch_dtype=inf_dtype,
            device_map="auto" if torch.cuda.is_available() else None
        )

        if adapter_path:
            print(f"🔀 Merging adapter from: {adapter_path}...")
            hf_model = PeftModel.from_pretrained(hf_model, adapter_path, token=HF_TOKEN)
            hf_model = hf_model.merge_and_unload()
        
        # Convert to CT2
        tmp_dir = "./tmp_merged_model"
        ct2_dir = "./tmp_ct2_model"
        if os.path.exists(tmp_dir): shutil.rmtree(tmp_dir)
        if os.path.exists(ct2_dir): shutil.rmtree(ct2_dir)
        
        print("💾 Saving merged model temporarily...")
        hf_model.save_pretrained(tmp_dir)
        processor.save_pretrained(tmp_dir)
        
        # Free HF model VRAM before CT2 load
        del hf_model
        torch.cuda.empty_cache()
        
        print("⚡ Converting to CTranslate2 format...")
        converter = ctranslate2.converters.TransformersConverter(tmp_dir)
        converter.convert(ct2_dir, force=True, quantization="float16")
        
        print("🚀 Loading into faster-whisper...")
        compute = "float16" if device == "cuda" else "int8"
        model = WhisperModel(ct2_dir, device=device, compute_type=compute)
        
        # Clean up temp files
        shutil.rmtree(tmp_dir)
    
    print("✅ Model loading complete.")

load_and_convert_model()


⏳ Loading native CT2 model from: C:\Users\marti\Projects\taigi_fast_whisper_finetuner\weights\whisper-medium-ct2-taigi-unified-part-3-r1-adapter-20260419-155233...
✅ Model loading complete.


In [ ]:
### Gradio UI

import gradio as gr
import numpy as np
import librosa

def to_srt(items, srt_poj_type):
    """Converts a list of items (Segments or Words) to SRT format."""
    def fmt(t):
        h, r = divmod(t, 3600)
        m, s = divmod(r, 60)
        ms = (s % 1) * 1000
        return f"{int(h):02}:{int(m):02}:{int(s):02},{int(ms):03}"

    lines = []
    for i, item in enumerate(items, 1):
        # faster_whisper.Segment has .text, TranscriptionWord has .word
        text = getattr(item, 'text', getattr(item, 'word', '')).strip()
        if srt_poj_type == "符號調" and POJ_AVAILABLE:
            try:
                text = poj_converter.convert_to_diacritic(text)
            except:
                pass
        lines.append(str(i))
        lines.append(f"{fmt(item.start)} --> {fmt(item.end)}")
        lines.append(text)
        lines.append("")
    return "\n".join(lines)

def transcribe_audio(audio_data, min_silence, speech_pad, threshold, min_speech, enable_max_speech, max_speech, timestamp_mode, generate_srt, srt_poj_type):
    """Transcribe audio data (filepath or 16kHz float32 numpy array)."""
    if audio_data is None:
        return "⚠️ No audio provided.", "", None, ""

    word_timestamps = (timestamp_mode == "音節")

    segments, info = model.transcribe(
        audio_data,
        vad_filter=True,
        vad_parameters=dict(
            min_silence_duration_ms=min_silence,
            speech_pad_ms=speech_pad,
            threshold=threshold,
            min_speech_duration_ms=min_speech,
            max_speech_duration_s=max_speech if enable_max_speech else float('inf')
        ),
        word_timestamps=word_timestamps,
        language=None  # Keeping as None per User Request "bypass bug 1"
    )

    segments = list(segments)

    full_text_list = []
    poj_text_list = []

    for segment in segments:
        if timestamp_mode == "語句":
            seg_text = f"[{segment.start:.2f}s --> {segment.end:.2f}s] {segment.text.strip()}"
            full_text_list.append(seg_text)
            if numeric_to_poj and POJ_AVAILABLE:
                try:
                    p_text = poj_converter.convert_to_diacritic(segment.text.strip())
                    poj_text_list.append(f"[{segment.start:.2f}s --> {segment.end:.2f}s] {p_text}")
                except:
                    poj_text_list.append(f"[{segment.start:.2f}s --> {segment.end:.2f}s] ⚠️ POJ Fail")
        elif timestamp_mode == "音節" and segment.words:
            for word in segment.words:
                word_text = f"[{word.start:.2f}s --> {word.end:.2f}s] {word.word.strip()}"
                full_text_list.append(word_text)
                if numeric_to_poj and POJ_AVAILABLE:
                    try:
                        p_text = poj_converter.convert_to_diacritic(word.word.strip())
                        poj_text_list.append(f"[{word.start:.2f}s --> {word.end:.2f}s] {p_text}")
                    except:
                        poj_text_list.append(f"[{word.start:.2f}s --> {word.end:.2f}s] ⚠️ POJ Fail")
        else:
            full_text_list.append(segment.text.strip())
            if numeric_to_poj and POJ_AVAILABLE:
                try:
                    p_text = poj_converter.convert_to_diacritic(segment.text.strip())
                    poj_text_list.append(p_text)
                except:
                    poj_text_list.append("⚠️ POJ Fail")

    full_text = "\n".join(full_text_list) if timestamp_mode != "無" else " ".join(full_text_list)
    poj_text = "\n".join(poj_text_list) if timestamp_mode != "無" else " ".join(poj_text_list)

    # Generate SRT
    srt_path = None
    srt_content = ""
    if generate_srt:
        if timestamp_mode == "音節":
            all_items = []
            for seg in segments:
                if seg.words:
                    all_items.extend(seg.words)
                else:
                    all_items.append(seg)
            srt_content = to_srt(all_items, srt_poj_type)
        else:
            srt_content = to_srt(segments, srt_poj_type)

        # Create the directory if it doesn't exist
        os.makedirs("./output", exist_ok=True) 

        srt_path = "./output/poj-output.srt"
        with open(srt_path, "w", encoding="utf-8") as f:
            f.write(srt_content)

    return full_text.strip(), poj_text.strip(), srt_path, srt_content

def run_fast_whisper_inference(mic_path, upload_path, min_silence, speech_pad, threshold, min_speech, enable_max_speech, max_speech, timestamp_mode, generate_srt, srt_poj_type):
    audio_path = mic_path if mic_path else upload_path
    if not audio_path:
        return "⚠️ No audio provided.", "", None, "", gr.update(visible=False)

    print(f"⏳ Transcribing with VAD (min_silence={min_silence}, pad={speech_pad}, threshold={threshold}, min_speech={min_speech}, max_speech={max_speech if enable_max_speech else 'inf'}, mode={timestamp_mode}).....")
    full_text, poj_text, srt_path, srt_content = transcribe_audio(audio_path, min_silence, speech_pad, threshold, min_speech, enable_max_speech, max_speech, timestamp_mode, generate_srt, srt_poj_type)

    if generate_srt and srt_path:
        return full_text, poj_text, srt_path, srt_content, gr.update(visible=True)
    else:
        return full_text, poj_text, None, "", gr.update(visible=False)

# --- UI Configuration Variables ---
ui_font_size = "35px"      # Controls labels, buttons, and UI text
output_font_size = "45px"  # Controls the transcribed output boxes

ui_texts = {
    "title": "🗣️ Fast-Whisper 台語推論",
    "model_info": "**Má-to͘h(Model):** {model_path} | **格式:** {model_format}",
    "timestamp_label": "時間標顯示:",
    "timestamp_choices": ["無", "語句", "音節"],
    "gen_srt_label": "Kám beh 生 .srt 檔案",
    "srt_poj_label": "SRT POJ 調號",
    "srt_poj_choices": ["數字調", "符號調"],
    "mic_label": "錄音",
    "upload_label": "ap-ló͘(upload)音聲檔",
    "clear_audio_btn": "🗑️ 音聲清hìⁿ-sak",
    "mic_btn": "🎙️ Mài-khuh(Mic)現錄",
    "upload_btn": "📁 用音聲檔",
    "vad_settings": "VAD 設定(自動切聲)",
    "min_silence_label": "上短無聲時間(毫秒)",
    "speech_pad_label": "語音chhun-kiu(毫秒)",
    "threshold_label": "VAD 濾坎",
    "min_speech_label": "上短語音時間(毫秒)",
    "enable_max_speech_label": "Kám beh 限制上長語音",
    "max_speech_label": "上長語音時間(秒)",
    "transcribe_btn": "✨ 現錄了a̍h-sī檔案chhoân好, 轉POJ",
    "raw_output_label": "數字調號輸出",
    "poj_output_label": "符號調號輸出",
    "srt_preview_label": "SRT 代先看",
    "srt_download_label": "táng-ló͘(download) SRT",
    "clear_text_btn": "🗑️ 文字清hìⁿ-sak"
}

# --- Build UI ---
custom_css = f"""
/* Layout constraints */
.audio-box {{ min-height: 200px !important; height: auto !important; max-height: none !important; padding-bottom: 20px !important; }}
.audio-box > div {{ min-height: 200px !important; height: auto !important; }}
.btn-mic-active {{ background-color: #ec4899 !important; color: white !important; border-color: #ec4899 !important; }}
.btn-upload-active {{ background-color: #3b82f6 !important; color: white !important; border-color: #3b82f6 !important; }}
.btn-inactive {{ background-color: #f3f4f6 !important; color: #374151 !important; border-color: #e5e7eb !important; }}

/* Constrain viewport width to ~2/3 and center it rigidly */
.gradio-container {{ max-width: 66% !important; width: 66% !important; margin: auto !important; overflow-x: hidden !important; }}

/* 1. Aggressive General UI Texts */
.gradio-container,
.gradio-container * {{
    font-size: {ui_font_size} !important;
}}

/* 2. Aggressive Transcribed Output Textboxes (Overrides General UI) */
.gradio-container .output-text,
.gradio-container .output-text *,
.gradio-container .output-text textarea,
.gradio-container .output-text input,
.gradio-container .output-text p,
.gradio-container .output-text div {{
    font-size: {output_font_size} !important;
    line-height: 1.5 !important;
    word-wrap: break-word !important;
    word-break: break-all !important;
    white-space: pre-wrap !important;
    overflow-wrap: break-word !important;
}}

/* 3. Keep Copy Button Size Specific */
.gradio-container .copy-btn,
.gradio-container .copy-btn * {{
    min-width: 40px !important;
    padding: 0 !important;
    font-size: 24px !important;
    align-self: flex-end !important;
}}
"""

with gr.Blocks(title=ui_texts["title"], css=custom_css) as demo:
    gr.Markdown(f"# {ui_texts['title']}")
    gr.Markdown(ui_texts["model_info"].format(model_path=model_path, model_format=model_format))

    with gr.Row():
        timestamp_radio = gr.Radio(choices=ui_texts["timestamp_choices"], value="無", label=ui_texts["timestamp_label"])
        generate_srt_checkbox = gr.Checkbox(label=ui_texts["gen_srt_label"], value=True)
        srt_poj_type_radio = gr.Radio(choices=ui_texts["srt_poj_choices"], value="數字調", label=ui_texts["srt_poj_label"], visible=True)     

    with gr.Row():
        with gr.Column(visible=True) as col_mic:
            mic_input = gr.Audio(sources=["microphone"], type="filepath", label=ui_texts["mic_label"], elem_classes=["audio-box"])        
        with gr.Column(visible=False) as col_upload:
            upload_input = gr.Audio(sources=["upload"], type="filepath", label=ui_texts["upload_label"], elem_classes=["audio-box"])

    with gr.Row():
        clear_audio_btn = gr.ClearButton(components=[mic_input, upload_input], value=ui_texts["clear_audio_btn"])

    with gr.Row():
        btn_mic = gr.Button(ui_texts["mic_btn"], elem_classes=["btn-mic-active"])
        btn_upload = gr.Button(ui_texts["upload_btn"], elem_classes=["btn-inactive"])

    with gr.Accordion(ui_texts["vad_settings"], open=False):
        min_silence = gr.Slider(500, 2000, value=1000, step=100, label=ui_texts["min_silence_label"])
        speech_pad = gr.Slider(100, 500, value=400, step=50, label=ui_texts["speech_pad_label"])
        threshold = gr.Slider(0.1, 0.9, value=0.5, step=0.1, label=ui_texts["threshold_label"])
        min_speech = gr.Slider(50, 1000, value=250, step=50, label=ui_texts["min_speech_label"])
        with gr.Row():
            enable_max_speech = gr.Checkbox(label=ui_texts["enable_max_speech_label"], value=True)
            max_speech = gr.Slider(5.0, 60.0, value=15.0, step=1.0, label=ui_texts["max_speech_label"])

    transcribe_btn = gr.Button(ui_texts["transcribe_btn"], variant="primary", visible=True)

    with gr.Row():
        with gr.Column(scale=9):
            raw_output = gr.Textbox(label=ui_texts["raw_output_label"], lines=3, elem_classes=["output-text"])
        with gr.Column(scale=1, min_width=50):
            raw_copy_btn = gr.Button("📋", elem_classes=["copy-btn"])
            
    with gr.Row():
        with gr.Column(scale=9):
            poj_output = gr.Textbox(label=ui_texts["poj_output_label"], lines=3, elem_classes=["output-text"])  
        with gr.Column(scale=1, min_width=50):
            poj_copy_btn = gr.Button("📋", elem_classes=["copy-btn"])

    with gr.Column(visible=False) as srt_section:
        with gr.Row():
            with gr.Column(scale=9):
                srt_preview = gr.Textbox(label=ui_texts["srt_preview_label"], lines=10, elem_classes=["output-text"])
            with gr.Column(scale=1, min_width=50):
                srt_copy_btn = gr.Button("📋", elem_classes=["copy-btn"])
        srt_download = gr.File(label=ui_texts["srt_download_label"])

    with gr.Row():
        clear_text_btn = gr.ClearButton(components=[raw_output, poj_output, srt_preview], value=ui_texts["clear_text_btn"])

    # --- UI Logic ---

    copy_js = "(text) => { navigator.clipboard.writeText(text); return text; }"
    raw_copy_btn.click(None, inputs=[raw_output], js=copy_js)
    poj_copy_btn.click(None, inputs=[poj_output], js=copy_js)
    srt_copy_btn.click(None, inputs=[srt_preview], js=copy_js)

    generate_srt_checkbox.change(
        fn=lambda gen: gr.update(visible=gen),
        inputs=[generate_srt_checkbox],
        outputs=[srt_poj_type_radio]
    )

    enable_max_speech.change(
        fn=lambda enable: gr.update(interactive=enable),
        inputs=[enable_max_speech],
        outputs=[max_speech]
    )

    def on_mic_click():
        return (
            gr.update(visible=True),  # col_mic
            gr.update(visible=False), # col_upload
            gr.update(elem_classes=["btn-mic-active"]),
            gr.update(elem_classes=["btn-inactive"]),
            gr.update(visible=False)  # srt_section
        )

    def on_upload_click():
        return (
            gr.update(visible=False), # col_mic
            gr.update(visible=True),  # col_upload
            gr.update(elem_classes=["btn-inactive"]),
            gr.update(elem_classes=["btn-upload-active"]),
            gr.update(visible=False)  # srt_section
        )

    btn_mic.click(
        fn=on_mic_click,
        outputs=[col_mic, col_upload, btn_mic, btn_upload, srt_section]
    )

    btn_upload.click(
        fn=on_upload_click,
        outputs=[col_mic, col_upload, btn_mic, btn_upload, srt_section]
    )

    # --- Inference Logic ---

    transcribe_btn.click(
        fn=run_fast_whisper_inference,
        inputs=[mic_input, upload_input, min_silence, speech_pad, threshold, min_speech, enable_max_speech, max_speech, timestamp_radio, generate_srt_checkbox, srt_poj_type_radio],
        outputs=[raw_output, poj_output, srt_download, srt_preview, srt_section]
    )

    clear_audio_btn.click(
        fn=lambda: gr.update(visible=False),
        outputs=[srt_section]
    )

    clear_text_btn.click(
        fn=lambda: gr.update(visible=False),
        outputs=[srt_section]
    )

demo.launch(share=not IS_LOCAL, inline=False, debug=True)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
⏳ Transcribing with VAD (min_silence=1000, pad=400, threshold=0.5, mode=無).....
⏳ Transcribing with VAD (min_silence=1000, pad=400, threshold=0.5, mode=語句).....
Keyboard interruption in main thread... closing server.
